# Sleep-EDF: refinement100–117

Порядок: prepare → gates → wav2sleep → temporal → blends → wav_blend → tiny → tiny_blends.
Первый запуск smoke; затем SMOKE_ONLY=False. Все настройки — в первой ячейке.
Подключите full_psg_v1 и подготовленный refinement-inputs.zip.019/082/085 заново не обучаются.

Общая подготовка создаёт nested OOF один раз для всех heads. Это отдельные12 внутренних fits, необходимые для отсутствия утечки приCV метамоделей. Кэшируйте и скачивайте их.
СтандартныеOOF082/085 используются для итогового refit, final experts — дляval/test.

После прерывания можно выполнить экспорт последней ячейкой. Результаты, веса и лёгкие кэши выгружаются тремяZIP; исходные сигналы вZIP не входят.


In [ ]:
import json
import os
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

REPO_URL = "https://github.com/revlze/sleep-edf-classification.git"
CODE_REF = "feat/pretrained-refinement"
PROJECT_ROOT = Path("/tmp/sleep-edf-refinement-code")
OUTPUT_ROOT = Path("/kaggle/working/refinement-results")
TRAINED_ROOT = Path("/kaggle/working/refinement-trained")
CACHE_ROOT = Path("/tmp/sleep-edf-refinement-cache")
CHECKPOINT_CACHE = Path("/tmp/sleep-edf-refinement-checkpoints")
CHECKPOINT_ROOT = None
INPUT_ROOTS = []  # Автопоиск refinement-inputs.zip или распакованного bundle.json.
FEATURE_CACHE = None  # Автопоиск106 признаков в том же Input bundle.
RESUME_ROOT = None  # Распакованные результаты НОВОЙ серии, для resume.
RESUME_TRAINED_ROOT = None  # Распакованные веса НОВОЙ серии.
RESUME_CACHE_ROOT = None  # Распакованный refinement-cache.zip: full/ и smoke/.

RUN_GROUP = "prepare"  # prepare / gates / wav2sleep / temporal / blends / wav_blend / tiny / tiny_blends / all / custom
CUSTOM_IDS = []
SMOKE_ONLY = True
SMOKE_EPOCHS = 160
ALLOW_TRAINING = True  # Нужен и для общей nested-подготовки.
DEVICE = "cuda:0"  # Одна GPU на процесс; VRAM двухT4 не складывается.
THREADS = min(4, os.cpu_count() or 2)
PARALLEL_DEVICES = [
    "cuda:0",
    "cuda:1",
]  # Только gates/temporal ПОСЛЕ prepare; [] для одной GPU.
EXPORT_TRAINED_CHECKPOINTS = True
EXPORT_REUSABLE_CACHE = True

GROUPS = {
    "prepare": [],
    "gates": list(range(100, 105)),
    "wav2sleep": [105],
    "temporal": list(range(106, 111)),
    "blends": [111, 112, 113],
    "wav_blend": [114],
    "tiny": [117],
    "tiny_blends": [115, 116],
    "all": [*range(100, 114), 114, 117, 115, 116],
}
CONFIG_IDS = (
    CUSTOM_IDS
    if RUN_GROUP == "custom"
    else [f"{n:03d}_pretrained" for n in GROUPS[RUN_GROUP]]
)
if RUN_GROUP != "prepare" and not CONFIG_IDS:
    raise ValueError("Нужен список полных experiment IDs")
if not ALLOW_TRAINING and (
    RUN_GROUP == "prepare"
    or any(int(n[:3]) in [*range(100, 105), *range(106, 111), 117] for n in CONFIG_IDS)
):
    raise ValueError("Для этой группы требуется ALLOW_TRAINING=True")


def checkout_code():
    if PROJECT_ROOT.exists():
        current_ref = subprocess.check_output(
            ["git", "config", "sleepedf.bootstrapRef"], cwd=PROJECT_ROOT, text=True
        ).strip()
        if current_ref != CODE_REF:
            raise RuntimeError(
                "Для другого CODE_REF задайте новый PROJECT_ROOT в /tmp."
            )
        print(
            "Используется существующий checkout; для обновления ветки задайте новый PROJECT_ROOT."
        )
        return
    from kaggle_secrets import UserSecretsClient

    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    with tempfile.TemporaryDirectory(prefix="sleep-git-") as temporary:
        askpass = Path(temporary) / "askpass.py"
        askpass.write_text(
            "#!/usr/bin/env python3\nimport os, sys\n"
            'print("x-access-token" if "Username" in sys.argv[1] else os.environ["SLEEP_GITHUB_TOKEN"])\n'
        )
        askpass.chmod(0o700)
        env = {
            **os.environ,
            "GIT_ASKPASS": str(askpass),
            "GIT_TERMINAL_PROMPT": "0",
            "SLEEP_GITHUB_TOKEN": token,
        }
        subprocess.run(
            ["git", "init", "--quiet", "--initial-branch=bootstrap", str(PROJECT_ROOT)],
            check=True,
        )
        subprocess.run(
            ["git", "remote", "add", "origin", REPO_URL], cwd=PROJECT_ROOT, check=True
        )
        subprocess.run(
            [
                "git",
                "-c",
                "credential.helper=",
                "fetch",
                "--quiet",
                "--depth",
                "1",
                "origin",
                CODE_REF,
            ],
            cwd=PROJECT_ROOT,
            env=env,
            check=True,
        )
        subprocess.run(
            ["git", "checkout", "--quiet", "--detach", "FETCH_HEAD"],
            cwd=PROJECT_ROOT,
            check=True,
        )
        subprocess.run(
            ["git", "config", "sleepedf.bootstrapRef", CODE_REF],
            cwd=PROJECT_ROOT,
            check=True,
        )
    del token


checkout_code()
commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True
).strip()
print("Код:", commit)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.notebook import reset_commands, run_command  # noqa: E402

reset_commands()

# Устанавливаем только инструмент uv; зависимости проекта устанавливает uv.lock.

UV_VERSION = "0.12.15"
if shutil.which("uv") is None:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", f"uv=={UV_VERSION}"], check=True
    )
UV = shutil.which("uv")
ENV = {
    "UV_PROJECT_ENVIRONMENT": "/tmp/sleep-edf-refinement-venv",
    "UV_CACHE_DIR": "/tmp/sleep-pretrained-uv-cache",
    "UV_PYTHON_INSTALL_DIR": "/tmp/sleep-pretrained-python",
    "PYTHONPATH": str(PROJECT_ROOT),
    "MPLBACKEND": "Agg",  # Backend только для дочернего CLI, не Kaggle-ядра.
}
UV_PROJECT = PROJECT_ROOT / "environments/pretrained"
print("Ядро notebook:", sys.version)
subprocess.run(["nvidia-smi"], check=True)
run_command(
    [
        UV,
        "sync",
        "--project",
        str(UV_PROJECT),
        "--locked",
        "--no-dev",
        "--python",
        "3.12.12",
    ],
    cwd=PROJECT_ROOT,
    env=ENV,
)
BASE = [
    UV,
    "run",
    "--project",
    str(UV_PROJECT),
    "--locked",
    "--no-dev",
    "python",
    "-m",
    "src.refinement_run",
]
ENV.update(
    {
        "OMP_NUM_THREADS": str(THREADS),
        "MKL_NUM_THREADS": str(THREADS),
        "OPENBLAS_NUM_THREADS": str(THREADS),
    }
)
run_command(
    [
        UV,
        "run",
        "--project",
        str(UV_PROJECT),
        "--locked",
        "--no-dev",
        "python",
        "-c",
        "from src.pretrained import hardware; import json; print(json.dumps(hardware("
        + repr(DEVICE)
        + "),indent=2))",
    ],
    cwd=PROJECT_ROOT,
    env=ENV,
)
print("Группа:", RUN_GROUP, "GPU:", DEVICE, "CPU threads:", THREADS)

In [ ]:
import hashlib
from zipfile import ZipFile

import pandas as pd
from IPython.display import display


def unpack(archive, target):
    target.mkdir(parents=True, exist_ok=True)
    marker = target / ".unpacked"
    if marker.exists():
        return target
    with ZipFile(archive) as zipped:
        for info in zipped.infolist():
            destination = (target / info.filename).resolve()
            if (
                not destination.is_relative_to(target.resolve())
                or (info.external_attr >> 16) & 0o170000 == 0o120000
            ):
                raise ValueError("Небезопасный путь в ZIP")
            zipped.extract(info, target)
    marker.touch()
    return target


roots = [
    p.parent
    for p in Path("/kaggle/input").rglob("MANIFEST.json")
    if (p.parent / "labels.csv").exists() and (p.parent / "epochs/train").is_dir()
]
if len(roots) != 1:
    raise RuntimeError(f"Нужен один full_psg_v1 dataset, найдено {roots}")
DATA_ROOT = roots[0]
if not INPUT_ROOTS:
    INPUT_ROOTS = sorted(
        set(p.parent for p in Path("/kaggle/input").rglob("bundle.json"))
    )
    for archive in Path("/kaggle/input").rglob("*.zip"):
        with ZipFile(archive) as zipped:
            names = zipped.namelist()
        if "bundle.json" in names:
            key = hashlib.sha256(
                (str(archive) + str(archive.stat().st_size)).encode()
            ).hexdigest()[:16]
            INPUT_ROOTS.append(unpack(archive, Path("/tmp/refinement-inputs") / key))
if not INPUT_ROOTS:
    raise FileNotFoundError(
        "Подключите sleep-edf-refinement-inputs.zip, подготовленный локально; повторять082/085 не требуется"
    )
if FEATURE_CACHE is None:
    definitions = [
        p.parent
        for r in INPUT_ROOTS
        for p in Path(r).rglob("definition.json")
        if json.loads(p.read_text()).get("version") == "linear106_v1"
        and not json.loads(p.read_text()).get("smoke")
    ]
    if len(definitions) == 1:
        FEATURE_CACHE = definitions[0]

# Копирование нужно только для resume, Input никогда не изменяется.
for previous, target in [
    (RESUME_ROOT, OUTPUT_ROOT),
    (RESUME_TRAINED_ROOT, TRAINED_ROOT),
    (RESUME_CACHE_ROOT, CACHE_ROOT),
]:
    if previous is not None:
        if target.exists():
            print("Существующий каталог сохранён, продолжаем:", target)
        else:
            shutil.copytree(Path(previous), target)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
COMMON = [
    "--data-root",
    str(DATA_ROOT),
    "--output-root",
    str(OUTPUT_ROOT),
    "--trained-root",
    str(TRAINED_ROOT),
    "--cache-root",
    str(CACHE_ROOT),
    "--checkpoint-cache",
    str(CHECKPOINT_CACHE),
    "--device",
    DEVICE,
    "--threads",
    str(THREADS),
]
for previous in INPUT_ROOTS:
    COMMON += ["--input-root", str(previous)]
if FEATURE_CACHE is not None:
    COMMON += ["--feature-cache", str(FEATURE_CACHE)]
if CHECKPOINT_ROOT is not None:
    COMMON += ["--checkpoint-root", str(CHECKPOINT_ROOT)]
if ALLOW_TRAINING:
    COMMON += ["--allow-training"]
for name in CONFIG_IDS:
    if not (PROJECT_ROOT / "configs" / (name + ".yaml")).exists():
        raise FileNotFoundError(name + ": нужен актуальныйCODE_REF")
display(
    pd.DataFrame(
        [
            {
                "ID": name,
                "status": "complete"
                if (
                    OUTPUT_ROOT
                    / ("smoke" if SMOKE_ONLY else "")
                    / name
                    / "metrics.json"
                ).exists()
                else "pending/partial",
            }
            for name in CONFIG_IDS
        ]
    )
)
print(
    "DATA_ROOT:",
    DATA_ROOT,
    "\nINPUT_ROOTS:",
    INPUT_ROOTS,
    "\nFEATURE_CACHE:",
    FEATURE_CACHE,
)
print(
    "Первое prepare: embeddings один раз и12 внутренних fits082/085 для корректнойCV. Это не12 новых глобальных экспериментов."
)

In [ ]:
def run_one(names, smoke, prepare=False, device=None, tag=None):
    import time

    started_ns = time.time_ns()
    command = [
        *BASE,
        *[str(PROJECT_ROOT / "configs" / (n + ".yaml")) for n in names],
        *COMMON,
    ]
    env = ENV.copy()
    if device is not None:
        command[command.index("--device") + 1] = device
        threads = max(1, min(THREADS, (os.cpu_count() or 2) // len(PARALLEL_DEVICES)))
        command[command.index("--threads") + 1] = str(threads)
        env.update(
            {
                k: str(threads)
                for k in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS")
            }
        )
    if tag is not None:
        command += ["--report-tag", tag]
    if smoke:
        command += ["--smoke", "--smoke-epochs", str(SMOKE_EPOCHS)]
    if prepare:
        command += ["--prepare"]
    try:
        run_command(command, cwd=PROJECT_ROOT, env=env)
    except subprocess.CalledProcessError:
        print(
            "Процесс сообщил об ошибках. Завершённые результаты сохранены; смотрите <ID>/last_error.json."
        )
        if prepare:
            raise
    namespace = OUTPUT_ROOT / "smoke" if smoke else OUTPUT_ROOT
    report = namespace / (
        f"refinement_failures-{tag}.json" if tag else "refinement_failures.json"
    )
    fresh = report.exists() and report.stat().st_mtime_ns >= started_ns
    failed = (
        {r["experiment_id"] for r in json.loads(report.read_text())}
        if fresh
        else set(names)
    )
    return [
        n
        for n in names
        if n not in failed and (namespace / n / "metrics.json").is_file()
    ]


def run_batch(names, smoke, prepare=False):
    from src.notebook import cancel_commands, reset_commands

    reset_commands()
    if prepare or RUN_GROUP not in ("gates", "temporal") or len(PARALLEL_DEVICES) < 2:
        return run_one(names, smoke, prepare)
    namespace = OUTPUT_ROOT / "smoke" if smoke else OUTPUT_ROOT
    if not (namespace / "prepare_report.json").exists():
        raise RuntimeError(
            "Для двух GPU сначала завершите prepare этого режима; либо PARALLEL_DEVICES=[]"
        )
    if len(PARALLEL_DEVICES) != 2 or len(set(PARALLEL_DEVICES)) != 2:
        raise ValueError(
            "PARALLEL_DEVICES должен содержать два разных устройства или быть []"
        )
    from concurrent.futures import ThreadPoolExecutor

    pool = ThreadPoolExecutor(max_workers=2)
    try:
        futures = [
            pool.submit(run_one, names[i::2], smoke, False, device, f"gpu{i}")
            for i, device in enumerate(PARALLEL_DEVICES)
            if names[i::2]
        ]
        passed = {name for future in futures for name in future.result()}
    except BaseException:
        cancel_commands()
        raise
    finally:
        pool.shutdown(wait=True, cancel_futures=True)
    return [name for name in names if name in passed]


if RUN_GROUP == "prepare":
    run_batch([], True, prepare=True)
    if not SMOKE_ONLY:
        run_batch([], False, prepare=True)
else:
    passed = run_batch(CONFIG_IDS, True)
    if not SMOKE_ONLY:
        run_batch(passed, False)
namespace = OUTPUT_ROOT / "smoke" if SMOKE_ONLY else OUTPUT_ROOT
rows = []
for name in CONFIG_IDS:
    p = namespace / name / "metrics.json"
    if p.exists():
        m = json.loads(p.read_text())
        rows.append(
            {
                "ID": name,
                "mode": "SMOKE" if m["smoke"] else "VAL+TEST",
                "val_macro_f1": m["validation"]["macro_f1"],
                "training_s": m["training_seconds"],
                "inference_s": m["inference_seconds"],
            }
        )
    else:
        e = namespace / name / "last_error.json"
        rows.append(
            {
                "ID": name,
                "mode": "FAILED/PARTIAL",
                "error": json.loads(e.read_text())["error"]
                if e.exists()
                else "not completed",
            }
        )
display(pd.DataFrame(rows))

Экспорт выполняется также после остановки обучения. Для продолжения в новой сессии распакуйте триZIP вInput и задайтеRESUME_ROOT, RESUME_TRAINED_ROOT, RESUME_CACHE_ROOT. Закрепите тот жеCODE_REF. Старые результаты сохраняются; smoke не попадает в experiments.csv.


In [ ]:
from datetime import UTC, datetime
from zipfile import ZIP_DEFLATED, ZipFile

from IPython.display import FileLink, display

stamp = datetime.now(UTC).strftime("%Y%m%dT%H%M%S%fZ")
archive = Path("/kaggle/working") / f"sleep-edf-refinement-results-{stamp}.zip"
run_command(
    [*BASE, "--output-root", str(OUTPUT_ROOT), "--export", str(archive)],
    cwd=PROJECT_ROOT,
    env=ENV,
)


def show(path, title):
    display(FileLink(str(path.relative_to(Path.cwd())), result_html_prefix=title))


show(archive, "Результаты и submissions: ")
if EXPORT_TRAINED_CHECKPOINTS and TRAINED_ROOT.exists():
    target = Path("/kaggle/working") / f"sleep-edf-refinement-trained-{stamp}.zip"
    with ZipFile(target, "x", compression=ZIP_DEFLATED, compresslevel=1) as zipped:
        for p in sorted(TRAINED_ROOT.rglob("*")):
            if (
                p.is_file()
                and p.suffix in (".pt", ".json", ".npz")
                and ".tmp." not in p.name
            ):
                zipped.write(p, p.relative_to(TRAINED_ROOT).as_posix())
    show(target, "Веса и partial inner predictions для resume: ")
if EXPORT_REUSABLE_CACHE and CACHE_ROOT.exists():
    target = Path("/kaggle/working") / f"sleep-edf-refinement-cache-{stamp}.zip"
    with ZipFile(target, "x", compression=ZIP_DEFLATED, compresslevel=1) as zipped:
        for p in sorted(CACHE_ROOT.rglob("*")):
            relative = p.relative_to(CACHE_ROOT)
            if (
                p.is_file()
                and p.suffix in (".json", ".npz")
                and not ({"signals", "pools"} & set(relative.parts))
                and ".tmp." not in p.name
            ):
                zipped.write(p, relative.as_posix())
    show(target, "Кэш embeddings/folds/nested OOF для следующей сессии: ")